# Public Notice Check (v2)
**A last check before a bilingual notice goes out.**
Veritas AI International High School Hackathon 2026 · Track: AI + Languages

A programme coordinator pastes the Traditional Chinese and English versions of a notice.
The tool flags sentences where the two versions may tell families different things
(when to apply, who qualifies, what to do), and says why. A person always makes the final call.

**What is pretrained and what is ours**
- Pretrained (not trained by us): **LaBSE** (matches sentences across languages) and **mDeBERTa NLI** (says whether one sentence contradicts another).
- Built by us: the number, negation and **date-role** checks, the logic that combines everything, the test data and the evaluation.

**How to run:** Runtime → Run all. The first run downloads two AI models (about 3 to 4 minutes).

## 1. Setup

In [ ]:
!pip -q install sentence-transformers gradio sentencepiece

import re
import itertools
import pandas as pd
import numpy as np
import torch

## 2. Data
Each row is one **sentence pair**.
- `has_error` = 1 if the English is wrong or misleading, 0 if it is a correct translation
- `error_type` = `none`, `number`, `negation`, `meaning`, `date_role` or `awkward`
- `notice_id` groups the correct and broken versions of the same sentence, so they always land in the same split

**Data lead:** replace these starter rows with your own sheet
(File → Download → CSV, upload it with the folder icon on the left, then uncomment the `read_csv` line).

In [ ]:
rows = [
  # notice_id, chinese, english, has_error, error_type
  (1, "報名截止日期為2026年12月5日。", "The application deadline is 5 December 2026.", 0, "none"),
  (1, "報名截止日期為2026年12月5日。", "The application deadline is 15 December 2026.", 1, "number"),
  (2, "本活動只限65歲或以上長者參加。", "This activity is only open to elderly people aged 65 or above.", 0, "none"),
  (2, "本活動只限65歲或以上長者參加。", "This activity is open to everyone aged 65 or below.", 1, "meaning"),
  (3, "參加者毋須繳付任何費用。", "Participants do not need to pay any fee.", 0, "none"),
  (3, "參加者毋須繳付任何費用。", "Participants need to pay a fee.", 1, "negation"),
  (4, "報名截止日期為11月20日，活動將於11月27日舉行。", "The application deadline is 20 November, and the activity will be held on 27 November.", 0, "none"),
  (4, "報名截止日期為11月20日，活動將於11月27日舉行。", "The application deadline is 27 November, and the activity will be held on 20 November.", 1, "date_role"),
  (5, "請於11月20日下午5時前交回表格。", "Please return the form by 5 p.m. on 20 November.", 0, "none"),
  (5, "請於11月20日下午5時前交回表格。", "Please return the form by 5 p.m. on 30 November.", 1, "number"),
  (6, "如遇八號或以上颱風信號，活動將會取消。", "If Typhoon Signal No. 8 or above is hoisted, the activity will be cancelled.", 0, "none"),
  (6, "如遇八號或以上颱風信號，活動將會取消。", "If Typhoon Signal No. 8 or above is hoisted, the activity will still go ahead.", 1, "meaning"),
  (7, "名額有限，先到先得。", "Places are limited and will be allocated on a first-come, first-served basis.", 0, "none"),
  (7, "名額有限，先到先得。", "The quota is limited, first arrive first get.", 1, "awkward"),
  (8, "學生必須由家長陪同出席。", "Students must be accompanied by a parent.", 0, "none"),
  (8, "學生必須由家長陪同出席。", "Students may attend without a parent.", 1, "negation"),
  (9, "查詢請致電2345 6789。", "For enquiries, please call 2345 6789.", 0, "none"),
  (9, "查詢請致電2345 6789。", "For enquiries, please call 2345 6798.", 1, "number"),
  (10, "參加者必須於活動當日帶備身份證明文件。", "Participants must bring an identity document on the day of the activity.", 0, "none"),
  (10, "參加者必須於活動當日帶備身份證明文件。", "Participants may bring an identity document on the day of the activity.", 1, "meaning"),
  (11, "只限香港居民申請。", "Only Hong Kong residents may apply.", 0, "none"),
  (11, "只限香港居民申請。", "Applications are open to everyone, including visitors.", 1, "meaning"),
  (12, "活動將於2026年12月12日（星期六）舉行。", "The activity will be held on Saturday, December 12, 2026.", 0, "none"),
  (12, "活動將於2026年12月12日（星期六）舉行。", "The activity will be held on Sunday, December 13, 2026.", 1, "number"),
  (13, "申請截止日期為12月1日，課程將於12月8日開始。", "Applications close on 1 December, and the course starts on 8 December.", 0, "none"),
  (13, "申請截止日期為12月1日，課程將於12月8日開始。", "Applications close on 8 December, and the course starts on 1 December.", 1, "date_role"),
]
df = pd.DataFrame(rows, columns=["notice_id", "chinese", "english", "has_error", "error_type"])

# df = pd.read_csv("notices.csv")   # <- use your own data here

# Split by notice: about 1/3 of notices for tuning (dev), the rest for the final score (test).
# Nothing is ever tuned on test rows.
ids = sorted(df["notice_id"].unique())
dev_ids = set(ids[: max(1, len(ids) // 3)])
df["split"] = df["notice_id"].apply(lambda i: "dev" if i in dev_ids else "test")
print(df["split"].value_counts())
df

split
test    18
dev      8
Name: count, dtype: int64


,notice_id,chinese,english,has_error,error_type,split
0,1,報名截止日期為2026年12月5日。,The application deadline is 5 December 2026.,0,none,dev
1,1,報名截止日期為2026年12月5日。,The application deadline is 15 December 2026.,1,number,dev
2,2,本活動只限65歲或以上長者參加。,This activity is only open to elderly people a...,0,none,dev
3,2,本活動只限65歲或以上長者參加。,This activity is open to everyone aged 65 or b...,1,meaning,dev
4,3,參加者毋須繳付任何費用。,Participants do not need to pay any fee.,0,none,dev
5,3,參加者毋須繳付任何費用。,Participants need to pay a fee.,1,negation,dev
6,4,報名截止日期為11月20日，活動將於11月27日舉行。,"The application deadline is 20 November, and t...",0,none,dev
7,4,報名截止日期為11月20日，活動將於11月27日舉行。,"The application deadline is 27 November, and t...",1,date_role,dev
8,5,請於11月20日下午5時前交回表格。,Please return the form by 5 p.m. on 20 November.,0,none,test
9,5,請於11月20日下午5時前交回表格。,Please return the form by 5 p.m. on 30 November.,1,number,test


## 3. Simplest baseline: a length rule
Flag a pair if the English is unusually long or short compared with the Chinese.
The allowed range is learned from the **correct** pairs in the dev split.

In [ ]:
def length_ratio(zh, en):
    return len(en.split()) / max(1, len(re.sub(r"\s", "", zh)))

dev_ok = df[(df.split == "dev") & (df.has_error == 0)]
ratios = [length_ratio(z, e) for z, e in zip(dev_ok.chinese, dev_ok.english)]
LOW, HIGH = min(ratios) * 0.8, max(ratios) * 1.2
print(f"Allowed English-words-per-Chinese-character range: {LOW:.2f} to {HIGH:.2f}")

def baseline_flag(zh, en):
    r = length_ratio(zh, en)
    return (r < LOW or r > HIGH), (["length looks off"] if (r < LOW or r > HIGH) else [])

Allowed English-words-per-Chinese-character range: 0.31 to 0.90


## 4. The pretrained models
- **LaBSE** turns a sentence in any language into a vector. Similar meaning → similar vectors. We use it to match sentences and to spot pairs that are about different things.
- **mDeBERTa NLI** reads two sentences and says whether the second one *follows from*, is *unrelated to*, or **contradicts** the first. We ask it in both directions (Chinese → English and English → Chinese) and keep the higher contradiction score.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)

NLI_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
nli_tok = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(device).eval()
# find which output position means "contradiction" from the model's own label list
CONTRA = [i for i, name in nli_model.config.id2label.items() if "contra" in name.lower()][0]
print("NLI labels:", nli_model.config.id2label)

def similarity(zh, en):
    a, b = labse.encode([zh, en], convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(a, b))

def contradiction(zh, en):
    # probability that the two sentences contradict each other (higher of the two directions)
    scores = []
    for premise, hypothesis in [(zh, en), (en, zh)]:
        inputs = nli_tok(premise, hypothesis, truncation=True, max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.softmax(nli_model(**inputs).logits[0], -1)
        scores.append(float(probs[CONTRA]))
    return max(scores)

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

... (19 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


## 5. Our rules
1. **Numbers:** every number in one version should appear in the other. Handles Chinese numerals (八 → 8) and English month names (December → 12).
2. **Negation:** if one version says "not / no / without" and the other doesn't, that's a red flag.
3. **Date roles (new):** a date is labelled as an **application deadline** or an **event date** from the words around it.
   If both versions contain the same dates but the *deadline* in one is the *event date* in the other, the plain number check sees nothing wrong. This check catches it.

In [ ]:
CN_DIGITS = {"零": 0, "〇": 0, "一": 1, "二": 2, "兩": 2, "三": 3, "四": 4, "五": 5, "六": 6, "七": 7, "八": 8, "九": 9}
CN_UNITS = {"十": 10, "百": 100, "千": 1000}
CN_NUM = "零〇一二兩三四五六七八九十百千"

def cn_to_int(s):
    if s.isdigit():
        return int(s)
    total, num = 0, 0
    for ch in s:
        if ch in CN_DIGITS:
            num = CN_DIGITS[ch]
        elif ch in CN_UNITS:
            total += (num or 1) * CN_UNITS[ch]
            num = 0
    return total + num

MONTH_NAMES = ["january", "february", "march", "april", "may", "june", "july",
               "august", "september", "october", "november", "december"]
MONTHS = {m: i + 1 for i, m in enumerate(MONTH_NAMES)}
EN_WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
            "nine": 9, "ten": 10, "eleven": 11, "twelve": 12}

def numbers_zh(text):
    text = re.sub(r"(星期|週|周|禮拜)[一二三四五六日天]", "", text)   # weekdays are not numbers
    nums = {int(n) for n in re.findall(r"\d+", text)}
    nums |= {cn_to_int(m) for m in re.findall(rf"[{CN_NUM}]+", text)}
    return nums

def numbers_en(text):
    t = text.lower()
    nums = {int(n) for n in re.findall(r"\d+", t)}
    nums |= {v for k, v in {**MONTHS, **EN_WORDS}.items() if k != "may" and re.search(rf"\b{k}\b", t)}
    if re.search(r"\d\s*may\b|\bmay\s*\d", t):   # "may" is only a month next to a number
        nums.add(5)
    return nums

NEG_ZH = re.compile(r"不|毋|無須|无须|未|沒|没|勿|別|非")
NEG_EN = re.compile(r"\b(not|no|never|without|cannot|none|nobody)\b|n't")

def has_neg_en(text):
    text = re.sub(r"\bNo\.\s*\d", "", text)   # "No. 8" means "number 8", not a negation
    return bool(NEG_EN.search(text.lower()))

In [ ]:
# ---- Date roles: is this date an application deadline or an event date? ----
DEADLINE_ZH = re.compile(r"截止|報名|申請|遞交|交回|提交|日前|時前|或之前")
EVENT_ZH = re.compile(r"舉行|舉辦|進行|開始|開課|上課")
DEADLINE_EN = re.compile(r"\b(deadline|apply|application|applications|submit|return|enrol|enroll|enrolment|"
                         r"enrollment|register|registration|close|closes|closing|by|before)\b")
EVENT_EN = re.compile(r"\b(held|take place|takes place|start|starts|begin|begins|commence|commences)\b")

MONTH_RE = "|".join(MONTH_NAMES)

def dates_in_zh(clause):
    return {(cn_to_int(m), cn_to_int(d)) for m, d in re.findall(rf"([\d{CN_NUM}]+)月([\d{CN_NUM}]+)[日號号]", clause)}

def dates_in_en(clause):
    t = clause.lower()
    found = {(MONTHS[m], int(d)) for d, m in re.findall(rf"(?<!\d)(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:of\s+)?({MONTH_RE})\b", t)}
    found |= {(MONTHS[m], int(d)) for m, d in re.findall(rf"\b({MONTH_RE})\s+(\d{{1,2}})(?!\d)", t)}
    return found

def role_of(clause, sentence, deadline_re, event_re):
    for text in (clause, sentence):          # look in the clause first, then the whole sentence
        if deadline_re.search(text):
            return "deadline"
        if event_re.search(text):
            return "event"
    return "other"

def date_roles(text, lang):
    roles = {"deadline": set(), "event": set(), "other": set()}
    if lang == "zh":
        sentences = [s for s in re.split(r"[。！？\n]", text) if s.strip()]
        for s in sentences:
            for clause in re.split(r"[，；、]", s):
                for d in dates_in_zh(clause):
                    roles[role_of(clause, s, DEADLINE_ZH, EVENT_ZH)].add(d)
    else:
        sentences = [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])|\n", text) if s.strip()]
        for s in sentences:
            for clause in re.split(r"[;]|,\s+(?=and\b|but\b|while\b)", s):
                for d in dates_in_en(clause):
                    roles[role_of(clause.lower(), s.lower(), DEADLINE_EN, EVENT_EN)].add(d)
    return roles

def show_dates(ds):
    return ", ".join(f"{d} {MONTH_NAMES[m - 1].title()}" for m, d in sorted(ds))

def date_role_reasons(zh, en):
    rz, re_ = date_roles(zh, "zh"), date_roles(en, "en")
    names = {"deadline": "application deadline", "event": "event date"}
    out = []
    for role, label in names.items():
        if rz[role] and re_[role] and rz[role] != re_[role]:
            out.append(f"{label} differs: Chinese says {show_dates(rz[role])}, English says {show_dates(re_[role])}")
    return out

def rule_reasons(zh, en):
    reasons = []
    nz, ne = numbers_zh(zh), numbers_en(en)
    if nz != ne:
        missing, extra = sorted(nz - ne), sorted(ne - nz)
        parts = ([f"{missing} only in Chinese"] if missing else []) + ([f"{extra} only in English"] if extra else [])
        reasons.append("numbers differ: " + "; ".join(parts))
    if bool(NEG_ZH.search(zh)) != has_neg_en(en):
        reasons.append("one version says 'not/no/without' and the other doesn't")
    reasons += date_role_reasons(zh, en)
    return reasons

# quick check of the new rule
print(date_role_reasons("報名截止日期為11月20日，活動將於11月27日舉行。",
                        "The application deadline is 27 November, and the activity will be held on 20 November."))

['application deadline differs: Chinese says 20 November, English says 27 November', 'event date differs: Chinese says 27 November, English says 20 November']


## 6. Score every pair once, then tune thresholds on the dev split only
The two model scores are computed once per row. Then we choose:
- `SIM_T`: flag if LaBSE similarity is **below** this
- `NLI_T`: flag if the contradiction probability is **above** this

**Tuning rule (same for every system):** on dev, keep settings where at least 90% of flags are real errors
(precision ≥ 0.9), then pick the one that catches the most errors. Test rows are never used here.

In [ ]:
df["similarity"] = [similarity(z, e) for z, e in zip(df.chinese, df.english)]
df["contradiction"] = [contradiction(z, e) for z, e in zip(df.chinese, df.english)]
df["rule_reasons"] = [rule_reasons(z, e) for z, e in zip(df.chinese, df.english)]
df[["split", "error_type", "similarity", "contradiction", "rule_reasons", "english"]].round(2)

,split,error_type,similarity,contradiction,rule_reasons,english
0,dev,none,0.94,0.01,[],The application deadline is 5 December 2026.
1,dev,number,0.86,1.00,[numbers differ: [5] only in Chinese; [15] onl...,The application deadline is 15 December 2026.
2,dev,none,0.90,0.02,[],This activity is only open to elderly people a...
3,dev,meaning,0.86,1.00,[],This activity is open to everyone aged 65 or b...
4,dev,none,0.91,0.00,[],Participants do not need to pay any fee.
5,dev,negation,0.73,1.00,[one version says 'not/no/without' and the oth...,Participants need to pay a fee.
6,dev,none,0.90,0.14,[],"The application deadline is 20 November, and t..."
7,dev,date_role,0.90,1.00,[application deadline differs: Chinese says 20...,"The application deadline is 27 November, and t..."
8,test,none,0.92,0.00,[],Please return the form by 5 p.m. on 20 November.
9,test,number,0.80,1.00,[numbers differ: [20] only in Chinese; [30] on...,Please return the form by 5 p.m. on 30 November.


In [ ]:
def model_reasons(sim, contra, sim_t, nli_t):
    reasons = []
    if sim < sim_t:
        reasons.append(f"meaning may differ (similarity {sim:.2f})")
    if contra > nli_t:
        reasons.append(f"the two versions may contradict each other ({contra:.0%})")
    return reasons

def tune(dev, use_rules):
    best, best_key = (0.0, 1.0), (-1, -1, -1)
    for sim_t, nli_t in itertools.product(np.arange(0.40, 0.96, 0.02), np.arange(0.30, 1.0, 0.05)):
        flags = [bool(model_reasons(s, c, sim_t, nli_t) or (use_rules and r))
                 for s, c, r in zip(dev.similarity, dev.contradiction, dev.rule_reasons)]
        flags = np.array(flags); err = dev.has_error.values == 1
        tp, fp = (flags & err).sum(), (flags & ~err).sum()
        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / err.sum()
        # reach 90% precision first, then most errors caught, then fewest false alarms
        key = (precision >= 0.9, recall, -fp)
        if key > best_key:
            best_key, best = key, (float(sim_t), float(nli_t))
    return best

dev = df[df.split == "dev"]
SIM_T_B, NLI_T_B = tune(dev, use_rules=False)   # B: pretrained models only
SIM_T, NLI_T = tune(dev, use_rules=True)        # C: full checker
print(f"B (models only): flag if similarity < {SIM_T_B:.2f} or contradiction > {NLI_T_B:.2f}")
print(f"C (full checker): flag if similarity < {SIM_T:.2f} or contradiction > {NLI_T:.2f}")

def checker_flag(zh, en, sim=None, contra=None):
    sim = similarity(zh, en) if sim is None else sim
    contra = contradiction(zh, en) if contra is None else contra
    reasons = rule_reasons(zh, en) + model_reasons(sim, contra, SIM_T, NLI_T)
    return bool(reasons), reasons

B (models only): flag if similarity < 0.40 or contradiction > 0.30
C (full checker): flag if similarity < 0.40 or contradiction > 0.30


## 7. Results on the test split
Four systems on the same held-out rows:
- **Baseline:** length rule
- **A. Rules only:** numbers, negation, date roles (no AI)
- **B. Pretrained models only:** LaBSE + mDeBERTa with thresholds, none of our rules
- **C. Full checker:** models + our rules

We report **errors caught** and **false alarms** separately, then break down by error type.

In [ ]:
test = df[df.split == "test"].copy()

systems = {
    "Baseline: length rule": lambda r: baseline_flag(r.chinese, r.english),
    "A. Rules only": lambda r: (bool(r.rule_reasons), r.rule_reasons),
    "B. Pretrained models only": lambda r: (lambda rs: (bool(rs), rs))(model_reasons(r.similarity, r.contradiction, SIM_T_B, NLI_T_B)),
    "C. Full checker": lambda r: checker_flag(r.chinese, r.english, r.similarity, r.contradiction),
}

summary = []
for name, fn in systems.items():
    out = test.apply(fn, axis=1)
    test[name] = [o[0] for o in out]
    test[name + " reasons"] = [o[1] for o in out]
    errs, oks = test[test.has_error == 1], test[test.has_error == 0]
    summary.append({
        "system": name,
        "errors caught": f"{int(errs[name].sum())} / {len(errs)}",
        "false alarms": f"{int(oks[name].sum())} / {len(oks)}",
    })
pd.DataFrame(summary)

,system,errors caught,false alarms
0,Baseline: length rule,0 / 9,1 / 9
1,A. Rules only,5 / 9,0 / 9
2,B. Pretrained models only,6 / 9,0 / 9
3,C. Full checker,7 / 9,0 / 9


In [ ]:
# Errors caught, broken down by error type
by_type = test[test.has_error == 1].groupby("error_type")[list(systems)].agg(lambda s: f"{int(s.sum())} / {len(s)}")
by_type

,Baseline: length rule,A. Rules only,B. Pretrained models only,C. Full checker
error_type,,,,
awkward,0 / 1,0 / 1,0 / 1,0 / 1
date_role,0 / 1,1 / 1,1 / 1,1 / 1
meaning,0 / 3,0 / 3,2 / 3,2 / 3
negation,0 / 1,1 / 1,1 / 1,1 / 1
number,0 / 3,3 / 3,2 / 3,3 / 3


In [ ]:
# Where it fails: what the full checker missed or wrongly flagged
full = "C. Full checker"
misses = test[(test.has_error == 1) & (~test[full])]
false_alarms = test[(test.has_error == 0) & (test[full])]
print("MISSED ERRORS")
display(misses[["error_type", "chinese", "english", "similarity", "contradiction"]])
print("FALSE ALARMS")
display(false_alarms[["chinese", "english", full + " reasons"]])

MISSED ERRORS


,error_type,chinese,english,similarity,contradiction
13,awkward,名額有限，先到先得。,"The quota is limited, first arrive first get.",0.773898,0.002111
19,meaning,參加者必須於活動當日帶備身份證明文件。,Participants may bring an identity document on...,0.841180,0.168701


FALSE ALARMS


,chinese,english,C. Full checker reasons


## 8. Check a whole notice
Sentences are split, each Chinese sentence is matched to its closest English sentence with LaBSE
(so reordered paragraphs still match), and every pair is checked. English sentences that match
nothing are flagged. Finally the **date roles are compared across the whole notice**, which catches
a swapped deadline even when the two dates sit in different sentences.

In [19]:
# Alignment and normalisation revision. Original model weights and thresholds.
from scipy.optimize import linear_sum_assignment
import unicodedata

SIM_T, NLI_T = 0.40, 0.30

def normalise_text(text, lang):
    text = unicodedata.normalize('NFKC', text)
    if lang == 'en':
        text = re.sub(r'\bnoon\b', '12 p.m.', text, flags=re.I)
        text = re.sub(r'\bmidnight\b', '12 a.m.', text, flags=re.I)
    return text

def numbers_en(text):
    text = normalise_text(text, 'en').lower()
    nums = {int(n) for n in re.findall(r'\d+', text)}
    words = {**EN_WORDS, 'both': 2, 'second': 2, 'first': 1}
    nums |= {v for k, v in words.items() if re.search(rf'\b{k}\b', text)}
    # Month names count only in explicit date phrases. Modal may reserve is excluded.
    for month, value in MONTHS.items():
        if re.search(rf'\b{month}\s+\d{{1,2}}\b|\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:of\s+)?{month}\b(?!\s+(?:reserve|apply|attend|submit|register|bring)\b)', text):
            nums.add(value)
    return nums

def numbers_zh(text):
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'(星期|週|周|禮拜)[一二三四五六日天]', '', text)
    text = re.sub(r'一併|一同|一起|一致|一般', '', text)
    return {int(n) for n in re.findall(r'\d+', text)} | {cn_to_int(n) for n in re.findall(rf'[{CN_NUM}]+', text)}

def negation_pair(zh, en):
    # Paired grammatical equivalents preserve the meaning of the complete clause.
    if re.search(r'\bunless\b', en, re.I):
        zh = zh.replace('除非', '')
    if re.search(r'\bunder\b|\bbelow\b|less than', en, re.I):
        zh = zh.replace('未滿', '')
    zh = zh.replace('不同', '').replace('不論', '').replace('不管', '')
    return bool(NEG_ZH.search(zh)), has_neg_en(en)

def clock_values(text, lang):
    text = normalise_text(text, lang)
    if lang == 'zh':
        text = re.sub(r'(上午|下午|中午|正午|晚上|凌晨)(\d{1,2})[時点點](至|到)(\d{1,2})[時点點]', lambda m: f'{m[1]}{m[2]}時{m[3]}{m[1]}{m[4]}時', text)
    values = []
    if lang == 'en':
        matches = re.findall(r'\b(\d{1,2})(?::(\d{2}))?\s*([ap])\.?m\.?', text, re.I)
        for hour, minute, half in matches:
            hour = int(hour)
            if 1 <= hour <= 12:
                values.append((hour % 12 + (12 if half.lower() == 'p' else 0)) * 60 + int(minute or 0))
    else:
        matches = re.findall(r'(上午|下午|中午|正午|晚上|凌晨)(\d{1,2})[時点點](?:(\d{1,2})分?)?', text)
        for half, hour, minute in matches:
            hour = int(hour)
            if hour <= 12:
                values.append((hour % 12 + (12 if half in ('下午','中午','正午','晚上') else 0)) * 60 + int(minute or 0))
    return set(values)

def rule_reasons(zh, en):
    reasons = []
    znum, enum = numbers_zh(zh), numbers_en(en)
    if znum != enum:
        reasons.append(f'numbers differ: {sorted(znum-enum)} only in Chinese; {sorted(enum-znum)} only in English')
    zclock, eclock = clock_values(zh, 'zh'), clock_values(en, 'en')
    if zclock and eclock and zclock != eclock:
        fmt = lambda values: ', '.join(f'{v//60:02d}:{v%60:02d}' for v in sorted(values))
        reasons.append(f'clock times differ: Chinese {fmt(zclock)}; English {fmt(eclock)}')
    zn, eneg = negation_pair(zh, en)
    if zn != eneg:
        reasons.append("one version says 'not/no/without' and the other doesn't")
    return reasons + date_role_reasons(zh, en)

def split_zh(text):
    # Keep conditions attached. Split independent availability statements after commas.
    return [s.strip() for s in re.split(r'(?<=[。！？；])|\n|(?<=，)(?=(?:不設|設有))', text) if s.strip()]

def split_en(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z])|(?<=;)\s*|\n', text) if s.strip()]

def alignment_text(text, lang):
    text = normalise_text(text, lang)
    # Avoid choosing a different action solely because its number happens to match.
    text = re.sub(r'\d+', ' ', text)
    return text

def checker_flag(zh, en, sim=None, contra=None):
    model_zh, model_en = normalise_text(zh, 'zh'), normalise_text(en, 'en')
    sim = similarity(model_zh, model_en) if sim is None else sim
    contra = contradiction(model_zh, model_en) if contra is None else contra
    reasons = rule_reasons(zh, en) + model_reasons(sim, contra, SIM_T, NLI_T)
    return bool(reasons), reasons

def check_notice(chinese_text, english_text):
    zh, en = split_zh(chinese_text), split_en(english_text)
    if not zh or not en:
        return pd.DataFrame([{'status':'⚠️ Check','chinese':chinese_text,'english':english_text,'why':'Paste both versions first.'}])
    ez = labse.encode([alignment_text(s,'zh') for s in zh], convert_to_tensor=True, normalize_embeddings=True)
    ee = labse.encode([alignment_text(s,'en') for s in en], convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(ez, ee).cpu().numpy()
    # Global assignment prevents two unrelated source sentences reusing one target.
    ii, jj = linear_sum_assignment(-sims)
    matches = dict(zip(ii.tolist(), jj.tolist()))
    used, out = set(), []
    for i, z in enumerate(zh):
        j = matches.get(i)
        if j is None or sims[i,j] < 0.25:
            out.append({'status':'⚠️ Check','chinese':z,'english':'','why':'Needs review: no confident English counterpart'})
            continue
        used.add(j)
        flagged, reasons = checker_flag(z, en[j])
        out.append({'status':'⚠️ Check' if flagged else '✅ OK','chinese':z,'english':en[j],'why':'; '.join(reasons)})
    for j, e in enumerate(en):
        if j not in used:
            out.append({'status':'⚠️ Check','chinese':'','english':e,'why':'Needs review: no confident Chinese counterpart'})
    for reason in date_role_reasons(chinese_text, english_text):
        if not any(reason in row['why'] for row in out):
            out.append({'status':'⚠️ Check','chinese':'(whole notice)','english':'(whole notice)','why':reason})
    return pd.DataFrame(out)


### Three test notices
1. **Clean pair:** a correct translation. Should be all ✅.
2. **Swapped dates:** both versions contain 20 and 27 November, but the deadline and event date are swapped in English. Should flag the deadline.
3. **Reordered and reformatted, still correct:** same meaning, sentences in a different order, date written differently. Should stay ✅ (this shows restraint, not just detection).

In [ ]:
fixture_zh = "報名截止日期為11月20日。活動將於11月27日舉行。參加者毋須繳付任何費用。"

fixtures = {
    "1. Clean pair (expect all OK)":
        "The application deadline is 20 November. The activity will be held on 27 November. Participants do not need to pay any fee.",
    "2. Swapped dates (expect a deadline warning)":
        "The application deadline is 27 November. The activity will be held on 20 November. Participants do not need to pay any fee.",
    "3. Reordered and reformatted (expect all OK)":
        "Participants do not need to pay any fee. The activity will be held on November 27. The application deadline is November 20.",
}
pd.set_option("display.max_colwidth", 120)
for name, english in fixtures.items():
    print("\n" + name)
    display(check_notice(fixture_zh, english))


1. Clean pair (expect all OK)


,status,chinese,english,why
0,✅ OK,報名截止日期為11月20日。,The application deadline is 20 November.,
1,✅ OK,活動將於11月27日舉行。,The activity will be held on 27 November.,
2,✅ OK,參加者毋須繳付任何費用。,Participants do not need to pay any fee.,



2. Swapped dates (expect a deadline warning)


,status,chinese,english,why
0,✅ OK,報名截止日期為11月20日。,The activity will be held on 20 November.,
1,⚠️ Check,活動將於11月27日舉行。,The activity will be held on 20 November.,"numbers differ: [27] only in Chinese; [20] only in English; event date differs: Chinese says 27 November, English sa..."
2,✅ OK,參加者毋須繳付任何費用。,Participants do not need to pay any fee.,
3,⚠️ Check,,The application deadline is 27 November.,no matching Chinese sentence: extra or changed content?
4,⚠️ Check,(whole notice),(whole notice),"application deadline differs: Chinese says 20 November, English says 27 November"



3. Reordered and reformatted (expect all OK)


,status,chinese,english,why
0,✅ OK,報名截止日期為11月20日。,The application deadline is November 20.,
1,✅ OK,活動將於11月27日舉行。,The activity will be held on November 27.,
2,✅ OK,參加者毋須繳付任何費用。,Participants do not need to pay any fee.,


## 9. Demo app

In [ ]:
import gradio as gr

gr.Interface(
    fn=check_notice,
    inputs=[gr.Textbox(label="Traditional Chinese version", lines=8, value=fixture_zh),
            gr.Textbox(label="English version", lines=8, value=fixtures["2. Swapped dates (expect a deadline warning)"])],
    outputs=gr.Dataframe(label="Findings", wrap=True),
    title="Public Notice Check",
    description="A last check before a bilingual notice goes out. It flags sentences to review; a person makes the final call.",
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0834d641946d7fb69.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Sources
- LaBSE: Feng et al., *Language-agnostic BERT Sentence Embedding* · https://huggingface.co/sentence-transformers/LaBSE
- mDeBERTa NLI: Laurer et al. · https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7 (MIT licence)
- sentence-transformers, transformers, PyTorch, pandas, NumPy, Gradio
- Test notices: written by our team (list any real public notices you add, with links)

In [ ]:
assert clock_values('上午9時至11時','zh') == {540,660}
old_results = [json.loads(line) for line in Path('/content/public_notice_v3_fixed_predictions.jsonl').read_text().splitlines()]
assert len(old_results)==60
by_id = {row['case_id']:row for row in old_results}
for case in benchmark_cases:
    if re.search(r'(上午|下午|中午|正午|晚上|凌晨)(\d{1,2})[時点點](至|到)(\d{1,2})[時点點]', case['chinese']):
        start=time.perf_counter()
        by_id[case['case_id']]={'case_id':case['case_id'],'rows':check_notice(case['chinese'],case['english']).to_dict('records'),'seconds':round(time.perf_counter()-start,3)}
        print('Recomputed',case['case_id'],flush=True)
Path('/content/public_notice_v3_final_predictions.jsonl').write_text(''.join(json.dumps(by_id[case['case_id']],ensure_ascii=False)+'\n' for case in benchmark_cases),encoding='utf-8')
print('Final 60 results saved',flush=True)
